# Charmonia Combined Analysis: CNM + Primordial

This notebook combines the results from the Cold Nuclear Matter (CNM) analysis (nPDF, energy loss, pT broadening) and the Primordial analysis (suppression in the medium) to produce the final $R_{pA}$ results.

The combination is performed as:
$$ R_{pA}^{\text{total}} = R_{pA}^{\text{CNM}} \times R_{pA}^{\text{Primordial}} $$

Uncertainties are propagated in quadrature:
$$ \delta_{\text{total}} = R_{pA}^{\text{total}} \sqrt{ \left( \frac{\delta_{\text{CNM}}}{R_{pA}^{\text{CNM}}} \right)^2 + \left( \frac{\delta_{\text{Prim}}}{R_{pA}^{\text{Prim}}} \right)^2 } $$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from dataclasses import dataclass

# --- Project Paths ---
project = Path.cwd()
sys.path.insert(0, str(project / "cnm_combine"))
sys.path.insert(0, str(project / "combined_code"))

# --- Modules ---
from cnm_combine import CNMCombine, combine_two_bands_1d
from primordial_module import (
    ReaderConfig, build_ensemble, make_bins_from_width, Style,
    Y_WINDOW_FORWARD, Y_WINDOW_BACKWARD, Y_WINDOW_CENTRAL
)

# --- Output Setup ---
outdir = project / "final_combined_output"
outdir.mkdir(exist_ok=True, parents=True)

# --- Plotting Style ---
Style.apply()
mpl.rcParams.update({
    "font.size": 11,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "legend.frameon": True,
    "legend.framealpha": 0.85,
    "axes.unicode_minus": False,
})

print("Modules loaded and output directory created.")

## Configuration

In [ ]:
# --- Analysis Parameters ---
ENERGY = "8.16"   # "8.16" or "5.02"
FAMILY = "charmonia"
STATES = ["jpsi_1S", "psi_2S"] # Add chicJ_1P if needed

# --- Binning ---
# Consistent binning for both CNM and Primordial
CENT_CLASSES = [(0, 10), (10, 20), (20, 40), (40, 60), (60, 80), (80, 100)]
Y_BINS = make_bins_from_width(-5.0, 5.0, 0.5)
PT_BINS = make_bins_from_width(0.0, 30.0, 1.0) # Finer bins for pT
PT_WINDOW_AVG = (0.0, 30.0)

# --- Primordial Input Paths ---
PRIM_PATHS = {
    "8.16": {
        "Pert":  project / "input/primordial/pPb8TeV/output_new_form/output_8pPb_Tf170_Pert",
        "NPWLC": project / "input/primordial/pPb8TeV/output_new_form/output_8pPb_Tf170_NPWLC",
    },
    "5.02": {
        "Pert":  project / "input/primordial/pPb5TeV/output_new_form/output_5pPb_Tf170_Pert",
        "NPWLC": project / "input/primordial/pPb5TeV/output_new_form/output_5pPb_Tf170_NPWLC",
    },
}
GLAUBER_PATHS = {
    "8.16": project / "input/glauber_data/8TeV",
    "5.02": project / "input/glauber_data/5TeV",
}

print(f"Configured for {ENERGY} TeV")

## 1. CNM Analysis

In [ ]:
# Initialize CNMCombine
cnm = CNMCombine.from_defaults(
    energy=ENERGY,
    family=FAMILY,
    cent_bins=CENT_CLASSES,
    y_edges=np.array([b[0] for b in Y_BINS] + [Y_BINS[-1][1]]),
    p_edges=np.array([b[0] for b in PT_BINS] + [PT_BINS[-1][1]]),
    pt_range_avg=PT_WINDOW_AVG
)

print("CNM Module Initialized")

## 2. Primordial Analysis

In [ ]:
@dataclass
class PrimordialCombo:
    energy: str
    model: str
    ens: object
    runs: dict

def load_primordial(energy, model):
    base = PRIM_PATHS[energy][model]
    glauber = GLAUBER_PATHS[energy]
    cfg = ReaderConfig(debug=False)
    
    try:
        ens, runs = build_ensemble(
            str(base),
            str(glauber),
            tags=("tau1", "tau2"),
            cfg=cfg,
            sqrts_NN=float(energy)
        )
        return PrimordialCombo(energy, model, ens, runs)
    except FileNotFoundError as e:
        print(f"[WARN] Missing primordial input: {e}")
        return None

# Load both models
prim_combos = {}
for model in ["Pert", "NPWLC"]:
    combo = load_primordial(ENERGY, model)
    if combo:
        prim_combos[model] = combo
        print(f"Loaded Primordial {model}")

## 3. Combination Logic

In [ ]:
def combine_results(cnm_res, prim_res, state):
    """
    Combine CNM and Primordial results.
    cnm_res: dict with keys 'val', 'lo', 'hi' (arrays)
    prim_res: dict with keys 'val', 'lo', 'hi' (arrays)
    """
    # Extract CNM
    R1_c = cnm_res['val']
    R1_lo = cnm_res['lo']
    R1_hi = cnm_res['hi']
    
    # Extract Primordial
    R2_c = prim_res['val']
    R2_lo = prim_res['lo']
    R2_hi = prim_res['hi']
    
    # Combine
    Rc, Rlo, Rhi = combine_two_bands_1d(
        R1_c, R1_lo, R1_hi,
        R2_c, R2_lo, R2_hi
    )
    
    return Rc, Rlo, Rhi

## 4. Compute and Plot: $R_{pA}$ vs Rapidity

In [ ]:
# --- Helpers for Primordial Aggregation ---
def get_primordial_y(combo, pt_window, y_bins, cent_classes, state):
    # Get raw per-b data
    dfC_b, dfB_b = combo.ens.central_and_band_vs_y_per_b(
        pt_window=pt_window,
        y_bins=y_bins,
        with_feeddown=True,
        use_nbin=True,
        flip_y=True
    )
    
    maps = next(iter(combo.runs.values())).centrality
    
    results = {}
    for (lo, hi) in cent_classes:
        tag = f"{int(lo)}-{int(hi)}%"
        # Use the internal aggregation function from primordial_module logic
        # We need to reimplement _aggregate_class or import it if it was public, 
        # but it's not exported in __all__. Let's define a local wrapper or use the one from the notebook snippet.
        # For safety/speed, I'll use the one defined in the notebook snippet logic.
        
        # Re-implementing _aggregate_class logic briefly here for robustness
        from primordial_module import _centrality_percent
        
        bs = np.sort(dfC_b["b"].unique())
        cents = _centrality_percent(maps, bs)
        sel_b = bs[(cents >= lo) & (cents < hi)]
        
        if sel_b.size == 0:
            results[tag] = None
            continue
            
        wvals = maps.b_to_nbin(sel_b)
        wvals = np.where((wvals > 0) & np.isfinite(wvals), wvals, 1.0)
        wvals = wvals / wvals.sum()
        wmap = dict(zip(sel_b, wvals))
        
        # Weighted mean for center
        dc = dfC_b[dfC_b["b"].isin(sel_b)].copy()
        rows = []
        for xv, chunk in dc.groupby("y", sort=True):
            ws = np.array([wmap[b] for b in chunk["b"]], float)
            val = np.sum(ws * chunk[state].to_numpy(float))
            rows.append({"y": float(xv), "val": val})
        res_c = pd.DataFrame(rows).sort_values("y")
        
        # Min/Max for band
        res_lo, res_hi = None, None
        if dfB_b is not None:
            db = dfB_b[dfB_b["b"].isin(sel_b)].copy()
            rows_b = []
            for xv, chunk in db.groupby("y", sort=True):
                lo_val = np.nanmin(chunk[f"{state}_lo"].to_numpy(float))
                hi_val = np.nanmax(chunk[f"{state}_hi"].to_numpy(float))
                rows_b.append({"y": float(xv), "lo": lo_val, "hi": hi_val})
            res_band = pd.DataFrame(rows_b).sort_values("y")
            res_lo = res_band["lo"].to_numpy()
            res_hi = res_band["hi"].to_numpy()
        else:
            res_lo = res_c["val"].to_numpy()
            res_hi = res_c["val"].to_numpy()
            
        results[tag] = {
            "y": res_c["y"].to_numpy(),
            "val": res_c["val"].to_numpy(),
            "lo": res_lo,
            "hi": res_hi
        }
    return results

# --- Calculation & Plotting ---
for state in STATES:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)
    axes = axes.flatten()
    
    # Get CNM results (same for all models)
    # cnm_vs_y returns a dict of DataFrames keyed by component
    # We need to extract per-centrality data
    # The cnm_vs_y method in CNMCombine aggregates internally based on cent_bins provided in init
    # Wait, cnm_vs_y returns a dict of arrays/dfs? Let's check the module.
    # It returns a dict: {component: DataFrame with columns [y, val, lo, hi] for each centrality?}
    # Actually cnm_vs_y returns a dict of DataFrames if we look at cnm_only.ipynb
    # But we need it per centrality class. 
    # The CNMCombine class has `cnm_vs_y` which takes `mb_weight_mode` etc.
    # It seems `cnm_vs_y` calculates for *one* centrality range if we don't specify otherwise?
    # No, looking at cnm_combine.py, `cnm_vs_y` calculates for the *whole* range or specific?
    # Ah, `cnm_vs_y` seems to return results for the *average* over the `cent_bins` if `include_mb` is True?
    # Let's re-read cnm_combine.py usage.
    # It seems we need to call it for each centrality bin if we want per-centrality plots.
    # OR, we can use `cnm_vs_y` which might return a dataframe with centrality columns?
    # Let's assume we need to loop over centrality classes for CNM as well.
    
    for i, (c_lo, c_hi) in enumerate(CENT_CLASSES):
        ax = axes[i]
        cent_tag = f"{c_lo}-{c_hi}%"
        
        # 1. Get CNM for this centrality
        # We temporarily set the object's cent_bins or we create a new one?
        # CNMCombine is designed to handle multiple bins? 
        # Actually, `cnm_vs_y` uses `self.gl.get_centrality_class` which uses `self.cent_bins`.
        # If `cent_bins` has multiple entries, does it return multiple? 
        # Let's assume we need to instantiate CNMCombine for each bin or use a method that supports it.
        # To be safe and simple: Instantiate CNMCombine for this specific bin.
        
        cnm_bin = CNMCombine.from_defaults(
            energy=ENERGY, family=FAMILY, 
            cent_bins=[(c_lo, c_hi)], # Single bin
            y_edges=np.array([b[0] for b in Y_BINS] + [Y_BINS[-1][1]]),
            pt_range_avg=PT_WINDOW_AVG
        )
        cnm_res_df = cnm_bin.cnm_vs_y(components=["cnm"])["cnm"] # Returns DF
        # Columns: y, val, lo, hi
        cnm_data = {
            "val": cnm_res_df["val"].values,
            "lo": cnm_res_df["lo"].values,
            "hi": cnm_res_df["hi"].values
        }
        y_axis = cnm_res_df["y"].values
        
        # Plot CNM
        ax.plot(y_axis, cnm_data["val"], color="gray", ls=":", label="CNM Only")
        ax.fill_between(y_axis, cnm_data["lo"], cnm_data["hi"], color="gray", alpha=0.2)
        
        # 2. Get Primordial & Combine for each model
        for model, color in [("Pert", "tab:blue"), ("NPWLC", "tab:red")]:
            if model not in prim_combos: continue
            
            # Get Primordial
            # We use our helper which returns a dict of results
            # We call it once per model, but here inside the loop is fine too (cached ideally)
            # For efficiency, let's call it outside? No, inside is fine for now.
            prim_all = get_primordial_y(prim_combos[model], PT_WINDOW_AVG, Y_BINS, [(c_lo, c_hi)], state)
            prim_data = prim_all[cent_tag]
            
            if prim_data is None: continue
            
            # Combine
            Rc, Rlo, Rhi = combine_results(cnm_data, prim_data, state)
            
            # Plot Combined
            ax.plot(y_axis, Rc, color=color, label=f"Total ({model})")
            ax.fill_between(y_axis, Rlo, Rhi, color=color, alpha=0.3)
            
        ax.set_title(f"Centrality {cent_tag}")
        ax.set_ylim(0, 1.2)
        ax.axhline(1, color="k", lw=0.5)
        
    axes[0].legend()
    fig.suptitle(f"{state} $R_{{pA}}$ vs $y$ at $\sqrt{{s_{{NN}}}} = {ENERGY}$ TeV")
    plt.tight_layout()
    plt.savefig(outdir / f"combined_rpa_vs_y_{state}.pdf")
    plt.show()

## 5. Compute and Plot: $R_{pA}$ vs Centrality

In [ ]:
# Similar logic for Centrality dependence
# We need to define Y-windows for this
Y_WINS = {"Forward": Y_WINDOW_FORWARD, "Backward": Y_WINDOW_BACKWARD, "Central": Y_WINDOW_CENTRAL}

for state in STATES:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    
    for i, (y_name, y_win) in enumerate(Y_WINS.items()):
        ax = axes[i]
        
        # CNM vs Centrality
        cnm_cent = CNMCombine.from_defaults(
            energy=ENERGY, family=FAMILY,
            cent_bins=CENT_CLASSES,
            y_windows=[(*y_win, y_name)],
            pt_range_avg=PT_WINDOW_AVG
        )
        # cnm_vs_centrality returns dict of DFs
        cnm_res = cnm_cent.cnm_vs_centrality(y_window=(*y_win, y_name), components=["cnm"])["cnm"]
        
        # Prepare X axis (midpoints of bins)
        cent_mids = [0.5*(lo+hi) for lo, hi in CENT_CLASSES]
        
        # Plot CNM
        ax.plot(cent_mids, cnm_res["val"], color="gray", ls=":", label="CNM Only")
        ax.fill_between(cent_mids, cnm_res["lo"], cnm_res["hi"], color="gray", alpha=0.2)
        
        # Primordial & Combine
        for model, color in [("Pert", "tab:blue"), ("NPWLC", "tab:red")]:
            if model not in prim_combos: continue
            
            # We need to compute primordial for each centrality bin and aggregate
            # Or use rpa_vs_b and aggregate?
            # Let's use the same logic as above: loop over bins, get value, append.
            
            prim_vals, prim_los, prim_his = [], [], []
            
            # We can use central_and_band_vs_y_per_b and aggregate, OR central_and_band_vs_b
            # central_and_band_vs_b gives vs b. We need to map to centrality classes.
            # Let's use the helper we wrote above but adapted for integrated y-window.
            
            # Actually, we can just call get_primordial_y with a single bin equal to the y-window?
            # No, get_primordial_y returns vs y.
            # We need integrated over y.
            
            # Let's do it per bin:
            for (c_lo, c_hi) in CENT_CLASSES:
                # Get vs y for this bin
                res_y = get_primordial_y(prim_combos[model], PT_WINDOW_AVG, [y_win], [(c_lo, c_hi)], state)
                tag = f"{int(c_lo)}-{int(c_hi)}%"
                data = res_y[tag]
                
                # Integrate over the single y-bin (which matches the window)
                # Since we passed [y_win] as bins, data['val'] should be an array of size 1
                if data is not None and len(data['val']) > 0:
                    prim_vals.append(data['val'][0])
                    prim_los.append(data['lo'][0])
                    prim_his.append(data['hi'][0])
                else:
                    prim_vals.append(np.nan)
                    prim_los.append(np.nan)
                    prim_his.append(np.nan)
            
            prim_vals = np.array(prim_vals)
            prim_los = np.array(prim_los)
            prim_his = np.array(prim_his)
            
            # Combine
            Rc, Rlo, Rhi = combine_two_bands_1d(
                cnm_res["val"].values, cnm_res["lo"].values, cnm_res["hi"].values,
                prim_vals, prim_los, prim_his
            )
            
            ax.plot(cent_mids, Rc, color=color, marker="o", label=f"Total ({model})")
            ax.fill_between(cent_mids, Rlo, Rhi, color=color, alpha=0.3)
            
        ax.set_title(f"{y_name} ({y_win[0]} < y < {y_win[1]})")
        ax.set_xlabel("Centrality (%)")
        ax.set_ylim(0, 1.2)
        ax.axhline(1, color="k", lw=0.5)
        
    axes[0].set_ylabel(r"$R_{pA}$")
    axes[0].legend()
    fig.suptitle(f"{state} $R_{{pA}}$ vs Centrality")
    plt.tight_layout()
    plt.savefig(outdir / f"combined_rpa_vs_cent_{state}.pdf")
    plt.show()